# Assessment 2: Machine Learning and Real-Time Streaming

**Student ID:** 35721588  
**Unit:** ITO5202  
**Teaching Period:** 5, 2026

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Source:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

In Assessment 1, we analysed historical freight charges across Olist shipping lanes. Now, in this task we want to build on that work using the same dataset. Firstly, in Part A, we look to train a model to predict the freight charged on each order item. Then, later in Part B, we want to be able to apply the saved model to orders it has not seen.

---

## Contents

**Setup**
- Environment and configuration
- Data loading
- Building the modelling dataset
- Train/stream split

**Part A: Machine learning pipeline**
1. ML task selection
2. Feature engineering
3. Model comparison and evaluation
4. Model persistence and verification

**Part B: Streaming, Kafka and reflection**
1. Streaming data simulation and Kafka producer
2. Structured Streaming consumer
3. Windowed aggregation
4. Reflection

---

## Environment and configuration

### Execution environment

Similar to what we did previously in Assessment 1, we want to run Spark in local mode on a single laptop, with its four logical cores acting as parallel task slots. The key difference for us now in this assessment is that a Kafka broker and ZooKeeper also run on the same machine during Part B. 

As we note in the below table, all three share the 8GB of memory. As such, Kafka's memory is capped at 512MB in order to leave enough room for Spark.

| Property | Value |
|---|---|
| Machine | MacBook Air (Retina, 13-inch, 2018) |
| Processor | 1.6 GHz dual-core Intel Core i5 |
| Logical cores | 4 |
| Physical memory | 8 GB |
| Operating system | macOS Sonoma 14.7.8 |
| Java | Eclipse Temurin JDK 17 (x64) |
| Python | 3.11.9 |
| PySpark | 3.5.1 |
| Kafka | 3.9.2 (ZooKeeper mode) |
| Spark master | `local[*]` |

In [2]:
# Import packages
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# File locations
DATA_DIR = "data/raw"
STREAM_PATH = "data/stream_data.parquet"
MODEL_PATH = "models/a2_model"

# Random seed for the train/stream split + model training
SEED = 42

### SparkSession configuration

In order for us to be able to complete the tasks needed for this assessment within our local set up, there were a few key Spark settings that we first made changes to. These are:

**`spark.driver.memory = 3g`**

Given that our notebook is running Spark in local mode, the driver is therefore doing the processing on the laptop itself. As such, we wanted to give Spark 3GB of memory rather than letting allowing for it to consume too much memory. 
This decision was made due to the fact that the laptop has 8 GB in available memory, and our later tasks in Part B need Kafka and ZooKeeper running alongside Spark. Because of this, leaving some memory free for those processes was critical.

**`spark.sql.shuffle.partitions = 4`**

Similar to our decision in Assessment 1, this setting was again set to 4. We note that the normal Spark default is 200 partitions, which is excessive for our dataset, which only has ~110,000 rows and would create lots of tiny tasks unneccesarily.

Furthermore, this decision also matters for our streaming section of this assessment. The shuffle partition setting is used by Spark when maintaining aggregation state, so using 200 partitions would mean each micro-batch has much more scheduling overhead. On the other hand, four partitions is more appropriate for the small local setup being used here.

**`spark.sql.session.timeZone = UTC`**

UTC is used here so that the timestamp handling stays consistent between runs rather than depending on the laptop's local time settings. This is particularly useful in Part B because the produced Kafka records contain timestamps.

It also avoids issues caused by Sydney changing between standard time and daylight saving time.

**`spark.jars.packages`**

We needed to add the Kafka connector separately due to the fact that it is not included in the normal PySpark installation. The connector version is matched to the Spark/PySpark version being used.

**Adaptive Query Execution**

AQE has not been manually disabled for this assessment, so Spark keeps its normal default behaviour.

In our previous assessment, AQE was turned off because the physical execution plans were being examined and needed to stay predictable. Given that this is not required here, there was no particular need for us to disable it. We note that AQE also does not apply to the streaming query itself, meaning it is only relevant to our batch work in Part A.

In [3]:
# Start building the Spark session
builder = SparkSession.builder

# Basic session settings
builder = builder.appName("ITO5202-A2-Olist-Freight")
builder = builder.master("local[*]")

# Memory and partition settings
builder = builder.config("spark.driver.memory", "3g")
builder = builder.config("spark.sql.shuffle.partitions", 4)

# Keep timestamps consistent
builder = builder.config("spark.sql.session.timeZone", "UTC")

# Add the Kafka connector
builder = builder.config(
    "spark.jars.packages",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
)

# Create the session
spark = builder.getOrCreate()

# Reduce console logging
spark.sparkContext.setLogLevel("ERROR")

spark

26/10/02 18:46:11 WARN Utils: Your hostname, Mounishas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.137 instead (on interface en0)
26/10/02 18:46:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/mmondeddu9/Desktop/ito5202-assessment2-streaming/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/mmondeddu9/.ivy2/cache
The jars for the packages stored in: /Users/mmondeddu9/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1de4b396-a890-4241-91e3-687f7ea492e9;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1248ms :: artifacts 

In [4]:
# Get the Spark context
sc = spark.sparkContext

# Check the main environment settings
print("Spark version:", spark.version)
print("Master:", sc.master)
print("Task slots:", sc.defaultParallelism)

# Check the configured values
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Time zone:", spark.conf.get("spark.sql.session.timeZone"))
print("Kafka package:", spark.conf.get("spark.jars.packages"))

Spark version: 3.5.1
Master: local[*]
Task slots: 4
Driver memory: 3g
Shuffle partitions: 4
AQE: true
Time zone: UTC
Kafka package: org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1


---

## Data loading

### Schemas

The seven CSV files that we need for our analysis are loaded with schemas written out manually instead of using `inferSchema=True`.

We note that it is particularly important for us to do this due to two key reasons:

1. Spark needs to inspect the data first when it infers a schema, which means another pass over the files before the actual load. This is unneccesary (and inefficient) for our use case, especially for the geolocation file which has over one million rows.

2. By using fixed schemas, we are also able to make the data types more predictable. Columns used across multiple tables, such as `order_id` and `product_id`, are defined the same way each time, which helps us to avoid type problems later on during joins.

We should also note again that the payments and reviews files are not used in this project. This is due to the fact that reviews only exist after an order has been delivered, and as such they would not be known, or relevant when freight is calculated. Also, payment values are also not used because they already include freight as part of the total amount paid. We consider these exclusions and our reasoning in more detail later in Part A.

### Correcting source column names

We note that there are two spelling mistakes in the original products CSV headers: `product_name_lenght` and `product_description_lenght`.

The initial schema still needs to use those exact names so Spark can read the file correctly. After the file is loaded, we can correct these column names.

In [5]:
# Schemas for the seven CSV files
order_items_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_item_id", IntegerType(), False),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

sellers_schema = StructType([
    StructField("seller_id", StringType(), False),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True)
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", IntegerType(), True),
    StructField("geolocation_lat", DoubleType(), True),
    StructField("geolocation_lng", DoubleType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True)
])

category_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True)
])


# Small helper for loading CSV files
def load_csv(filename, schema):
    file_path = DATA_DIR + "/" + filename

    df = spark.read \
        .option("header", "true") \
        .schema(schema) \
        .csv(file_path)

    return df


# Load each dataset
order_items = load_csv("olist_order_items_dataset.csv", order_items_schema)
orders = load_csv("olist_orders_dataset.csv", orders_schema)
customers = load_csv("olist_customers_dataset.csv", customers_schema)
sellers = load_csv("olist_sellers_dataset.csv", sellers_schema)
geolocation = load_csv("olist_geolocation_dataset.csv", geolocation_schema)
categories = load_csv("product_category_name_translation.csv", category_schema)

# Load products separately so the headers can be fixed
products = load_csv("olist_products_dataset.csv", products_schema)

# Correct the two spelling mistakes
products = products.withColumnRenamed(
    "product_name_lenght",
    "product_name_length"
)

products = products.withColumnRenamed(
    "product_description_lenght",
    "product_description_length"
)

In [6]:
# Store each dataset with its expected row count
datasets = [
    ("order_items", order_items, 112650),
    ("orders", orders, 99441),
    ("customers", customers, 99441),
    ("sellers", sellers, 3095),
    ("geolocation", geolocation, 1000163),
    ("products", products, 32951),
    ("categories", categories, 71)
]

# Build a simple validation table
rows = []

for name, df, expected in datasets:
    actual = df.count()
    number_of_columns = len(df.columns)

    row = {
        "dataset": name,
        "expected": expected,
        "actual": actual,
        "match": actual == expected,
        "columns": number_of_columns
    }

    rows.append(row)

# Display the results
count_check = pd.DataFrame(rows)

count_check

,dataset,expected,actual,match,columns
0,order_items,112650,112650,True,7
1,orders,99441,99441,True,8
2,customers,99441,99441,True,5
3,sellers,3095,3095,True,4
4,geolocation,1000163,1000163,True,5
5,products,32951,32951,True,9
6,categories,71,71,True,2


---

## Building the modelling dataset

The data that we need for our modelling comes from several separate files within the Olist dataset, so we need to first join them into one dataset with one row for each order item. This is also the level we want to use later for prediction due to the fact that in our data, freight is recorded against individual items rather than only at the order level.

While our overall set up is quite similar to what we did in Assessment 1, there are a few key differences we should note:

**Keep the original inputs**

In Assessment 1, we calculated some extra fields such as distance and package volume while our main dataset was being built. Here however, we want to instead keep the original values, such as the seller and customer coordinates and the package dimensions.

While we still want to do these calculations, we instead move them into the feature engineering pipeline later in our analysis. Fundamentally, this also makes the batch and streaming components easier to keep consistent, due to the fact that the Kafka records can contain the same raw fields and the saved pipeline can apply the same transformations to them.

**Add the number of items in each order**

We want to add a count of the items in each order to every row. This information is already available when the order is placed and may be useful because orders containing more items can have freight split differently.

**Do not create features using `freight_value`**

Some of the measures we used in Assessment 1, such as freight per kilogram and freight-to-price ratio, use `freight_value` in their calculation.

We note that we cannot do that here due t `freight_value` being the target that is being predicted. Using those fields as model inputs would leak information about the answer into the model.

**Keep missing values at this stage**

Rows with missing product, category or location values are not removed while building the dataset and instead are dealt with later inside the modelling pipeline.

This is important for us later on in Part B as well, since incoming streaming records may also have missing fields and should go through the same preparation steps as the training data.

Finally, we note that only delivered orders with a delivery date are kept, which matches our filtering decisioning used in Assessment 1. More specifically, cancelled and unavailable orders are left out because due to the fact that they did not result in a completed delivery.

### Preparing the source tables

Similar to what we did in Assessment 1, there are some important data cleaning steps that we need to take in order to ensure our datasets are usable for our analysis.

**Orders**

Our orders table is filtered to delivered orders that also have a delivery date recorded. This keeps our modelling data focused on completed shipments only without noise from cancelled orders.

**Geolocation**

Firstly, our geolocation file has multiple latitude and longitude records for the same postcode prefix. If we were to join this directly, a single order item could match several location rows and get duplicated.

To avoid that, the coordinates are averaged so there is only one row for each postcode prefix. This smaller lookup table is cached so it can be leveraged for later analysis.

**Category translations**

Our category translation file is also checked before joining. Like we noted in the previous assessment, there are Portuguese category names which do not have an English translation in the supplied file. As such, these translations are added manually.

After we have do our preparation, we are able to trim them down to only the columns needed for the joins and the model.

In [7]:
# Average coordinates for each postcode prefix
geo_lookup = geolocation.groupBy("geolocation_zip_code_prefix") \
    .agg(
        F.avg("geolocation_lat").alias("lat"),
        F.avg("geolocation_lng").alias("lng")
    )

# Cache because this lookup is used twice later
geo_lookup.cache()

geo_rows = geo_lookup.count()
print("Geolocation lookup rows:", geo_rows)


# Keep completed deliveries with a delivery date
orders_delivered = orders.filter(
    F.col("order_status") == "delivered"
)

orders_delivered = orders_delivered.filter(
    F.col("order_delivered_customer_date").isNotNull()
)

delivered_rows = orders_delivered.count()
print("Delivered orders retained:", delivered_rows)


# Add the two missing category translations
extra_categories = [
    ("pc_gamer", "pc_gamer"),
    (
        "portateis_cozinha_e_preparadores_de_alimentos",
        "kitchen_portables_and_food_preparers"
    )
]

missing_categories = spark.createDataFrame(
    extra_categories,
    ["product_category_name", "product_category_name_english"]
)

# Combine them with the supplied translations
categories_complete = categories.unionByName(missing_categories)

Geolocation lookup rows: 19015


Delivered orders retained: 96470


In [8]:
# Keep the order fields needed later
orders_join = orders_delivered.select(
    "order_id",
    "customer_id",
    "order_purchase_timestamp"
)

# Keep the customer location fields
customers_join = customers.select(
    "customer_id",
    "customer_state",
    "customer_zip_code_prefix"
)

# Keep the seller location fields
sellers_join = sellers.select(
    "seller_id",
    "seller_state",
    "seller_zip_code_prefix"
)

# Keep the product fields used for modelling
products_join = products.select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)


# Make a customer version of the postcode lookup
customer_geo = geo_lookup.select(
    F.col("geolocation_zip_code_prefix").alias("customer_zip_code_prefix"),
    F.col("lat").alias("customer_lat"),
    F.col("lng").alias("customer_lng")
)

# Make a seller version of the same lookup
seller_geo = geo_lookup.select(
    F.col("geolocation_zip_code_prefix").alias("seller_zip_code_prefix"),
    F.col("lat").alias("seller_lat"),
    F.col("lng").alias("seller_lng")
)

### Joining the tables

Our order item table is used as the main starting point, and the other tables are joined onto it.

Importantly, we want to use inner joins for our rrders, customers and sellers utables, due to the fact that each retained item should belong to a delivered order and have a matching customer and seller.

On the other hand, we instead use left joins for products, category translations and postcode locations in order to ensure that an order item is not removed simply because one of those extra fields is missing.

Furthermore, we want to ensure that the smaller lookup tables, such as sellers, products, categories and the postcode coordinate tables, are broadcast before joining in order to avoid moving the larger order data around as much during the joins.

Finally, the total number of items in each order is also calculated from the original order items table. That count is then joined back onto each individual item so every row includes the size of its order.

In [10]:
# Count how many items are in each order
order_sizes = order_items.groupBy("order_id") \
    .agg(F.count("*").alias("order_item_count"))


# Start with the main order item data
model_df = order_items.join(
    orders_join,
    on="order_id",
    how="inner"
)

# Add customer details
model_df = model_df.join(
    customers_join,
    on="customer_id",
    how="inner"
)

# Add seller details
model_df = model_df.join(
    F.broadcast(sellers_join),
    on="seller_id",
    how="inner"
)

# Add basket size
model_df = model_df.join(
    order_sizes,
    on="order_id",
    how="inner"
)


# Add product information
model_df = model_df.join(
    F.broadcast(products_join),
    on="product_id",
    how="left"
)

# Add translated category names
model_df = model_df.join(
    F.broadcast(categories_complete),
    on="product_category_name",
    how="left"
)

# Add customer coordinates
model_df = model_df.join(
    F.broadcast(customer_geo),
    on="customer_zip_code_prefix",
    how="left"
)

# Add seller coordinates
model_df = model_df.join(
    F.broadcast(seller_geo),
    on="seller_zip_code_prefix",
    how="left"
)


# Keep only the fields needed for modelling
model_df = model_df.select(
    "order_id",
    "order_item_id",
    "order_purchase_timestamp",
    "freight_value",
    "price",
    "order_item_count",
    "seller_state",
    "customer_state",
    "seller_lat",
    "seller_lng",
    "customer_lat",
    "customer_lng",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

# Check the final size
row_count = model_df.count()
column_count = len(model_df.columns)

print("Rows:", row_count)
print("Columns:", column_count)

Rows: 110189
Columns: 17


### Checking and caching the modelling dataset

Before we are able to use the dataset, we want to first check the joins to make sure each order item still appears only once. That is to say, we want to ensure that no duplicates were inadvertantly created.

Once our checks are completed, our completed modelling table is then cached as it is used several times later in the notebook, including for the target checks, splitting the data and training the models. By caching it, we are thus able to avoid running all of the joins again each time the dataset is used.

Finally, we want to conduct a check for any missing values. We want to be able to count these across the columns so that we are able to construct a quick view of which fields contain gaps and therefore need to be dealt with later in the feature pipeline.

In [11]:
# Check that order items were not duplicated
total_rows = model_df.count()

distinct_items = model_df.select(
    "order_id",
    "order_item_id"
).distinct().count()

print("Rows:", total_rows)
print("Distinct order items:", distinct_items)


# Cache the modelling dataset for reuse
model_df.cache()

# Run an action so the cache is populated
cached_rows = model_df.count()

print("Cached rows:", cached_rows)

Rows: 110189
Distinct order items: 110189


Cached rows: 110189


In [12]:
# Count nulls in each column
missing_counts = []

for column in model_df.columns:
    missing_count = model_df.filter(
        F.col(column).isNull()
    ).count()

    missing_counts.append({
        "column": column,
        "missing_rows": missing_count
    })

# Put the results into a small table
missing = pd.DataFrame(missing_counts)

missing

,column,missing_rows
0,order_id,0
1,order_item_id,0
2,order_purchase_timestamp,0
3,freight_value,0
4,price,0
5,order_item_count,0
6,seller_state,0
7,customer_state,0
8,seller_lat,249
9,seller_lng,249


#### Findings

We can see from our checks that the joins did not create any duplicate order items. There are 110,189 rows in the modelling dataset and 110,189 distinct combinations of `order_id` and `order_item_id`, which is consistent with our base dataset from Assessment 1.

The main identifiers that we are interested in for our analysis, `freight_value` target and order-level fields are complete. Any missing data is limited to a few fields:

- **Coordinates:** 249 rows are missing seller coordinates and 288 are missing customer coordinates.
- **Product category:** 1,537 rows do not have an English product category, which is about 1.4% of the rows.
- **Product measurements:** 18 rows are missing the product weight, length, height and width. Because the counts are the same across all four fields, these are most likely to be the same products with all physical measurements missing.

We should note that our feature pipeline in Part A, Section 2 handles these missing values before the models are trained, and as such we do not need to address any missing data at this stage.

---

## Train/stream split

Before we do any model training or analysis, we first need to split our modelling data into two components:

- **Training data (70%):** We use the training set in Part A to explore the data, train our models and evaluate their performance.
- **Streaming data (30%):** Our streaming set is kept separate and used later in Part B when our records are sent through Kafka.

We note that the streaming portion is not used in any way during model training or evaluation. This ensures that the records we use in Part B are unseen by the model beforehand.

**Split by order**

Since `order_id` values are unique, we are able to use that to perform the split of the two components rather than using individual order-item rows. We are able to apply `randomSplit([0.7, 0.3], seed=42)` to the orders first, and then all items belonging to an order are put into the same subset.

This is particularly important for us because items from the same order share a lot of information, such as the customer, seller, route and purchase date. As such, if items were split individually, one item from an order could appear in training while another item from the same order appeared in the held-out data. That would result in the two datasets being less independent and could make model performance look better than it really is.

**Fixed seed**

As we noted above, a seed of 42 is used for the split. This is done so that the same orders are assigned to each subset when the notebook is rerun with the same data.

In [13]:
# Get the unique orders first
distinct_orders = model_df.select("order_id").distinct()

# Split orders into training and streaming groups
train_orders, stream_orders = distinct_orders.randomSplit(
    [0.7, 0.3],
    seed=SEED
)

# Join the order groups back to the item-level data
train_df = model_df.join(
    train_orders,
    on="order_id",
    how="inner"
)

stream_df = model_df.join(
    stream_orders,
    on="order_id",
    how="inner"
)

# Cache both datasets for later use
train_df.cache()
stream_df.cache()

DataFrame[order_id: string, order_item_id: int, order_purchase_timestamp: timestamp, freight_value: double, price: double, order_item_count: bigint, seller_state: string, customer_state: string, seller_lat: double, seller_lng: double, customer_lat: double, customer_lng: double, product_category_name_english: string, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int]

In [14]:
# Count rows in each subset
train_items = train_df.count()
stream_items = stream_df.count()

# Count unique orders in each subset
train_order_count = train_df.select("order_id").distinct().count()
stream_order_count = stream_df.select("order_id").distinct().count()


# Check that no order appears in both groups
train_order_ids = train_df.select("order_id").distinct()
stream_order_ids = stream_df.select("order_id").distinct()

overlap = train_order_ids.join(
    stream_order_ids,
    on="order_id",
    how="inner"
).count()


# Put the split results into a table
split_rows = [
    {
        "subset": "training",
        "orders": train_order_count,
        "items": train_items
    },
    {
        "subset": "streaming",
        "orders": stream_order_count,
        "items": stream_items
    }
]

split_summary = pd.DataFrame(split_rows)

# Calculate percentage shares
total_orders = split_summary["orders"].sum()
total_items = split_summary["items"].sum()

split_summary["order_share_%"] = (
    split_summary["orders"] / total_orders * 100
).round(2)

split_summary["item_share_%"] = (
    split_summary["items"] / total_items * 100
).round(2)


# Final checks
print("Total items:", train_items + stream_items, "(expected 110,189)")
print("Orders in both subsets:", overlap)

split_summary

Total items: 110189 (expected 110,189)
Orders in both subsets: 0


,subset,orders,items,order_share_%,item_share_%
0,training,67726,77363,70.2,70.21
1,streaming,28744,32826,29.8,29.79


**Split results**

We can see from the above that our 96,470 delivered orders were split into 67,726 training orders and 28,744 streaming orders. This gives us roughly 70.2% of orders to training and 29.8% to streaming.

At an item level, the training set contains 77,363 rows and the streaming set contains 32,826 rows, which gives proportions that are very similar to the order-level split, meaning that grouping the split by order did not noticeably impact the number of items in either subset.

Together, the two subsets contain all 110,189 order items and all 96,470 orders. Our overlap check above also returned zero, meaning that that no order appears in both datasets.

### Saving our streaming set

Our streaming data is saved to `data/stream_data.parquet` so that we can load it up again using the Kafka producer in Part B later.

Since this subset is only around 33,000 rows, it can be written as a single data file and thus allows us to keep the output simple. We use overwrite mode here so that the notebook can be rerun without needing to manually delete the old file first, which important for us as we test and iterate on our analysis.

In saying that however, given that we used a fixed seed for the split, rerunning the notebook with the same source data should produce the same streaming records regardless.

In [15]:
# Save the streaming data as one parquet file
stream_output = stream_df.coalesce(1)

stream_output.write \
    .mode("overwrite") \
    .parquet(STREAM_PATH)


# Read the saved file back in
stream_check = spark.read.parquet(STREAM_PATH)

# Check the saved output
saved_rows = stream_check.count()
saved_columns = len(stream_check.columns)

print("Rows saved:", saved_rows)
print("Columns saved:", saved_columns)

stream_check.printSchema()

Rows saved: 32826
Columns saved: 17
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- price: double (nullable = true)
 |-- order_item_count: long (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- seller_lat: double (nullable = true)
 |-- seller_lng: double (nullable = true)
 |-- customer_lat: double (nullable = true)
 |-- customer_lng: double (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

